In [1]:
# ============================================================
# FINAL PCA VARIANCE + APPROXIMATE BACK-PROJECTION
# PCA TRAIN-ONLY FINAL VERSION
# ============================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

BASE_DIR = Path(".").resolve()

OUTCOME_DIRS = {
    "victimization": BASE_DIR / "final_victim/DT_victim_PCA_trainonly_TEST",
    "perpetration": BASE_DIR / "final_perpetrator/content/perpetrator_v3",
    "overlap": BASE_DIR / (
        "final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42"
    ),
}

FEATURES_PATH = BASE_DIR / "data/lista_global_vars.csv"

OUTPUT_DIR = BASE_DIR / "final_pca_variance_backprojection_PCA_trainonly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N_LOADINGS_PER_PC = 8

print("BASE_DIR:", BASE_DIR)
print("FEATURES_PATH:", FEATURES_PATH, FEATURES_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def find_pca_file(outcome_dir):
    outcome_dir = Path(outcome_dir)

    candidates = []
    patterns = [
        "*pca*.pkl",
        "*PCA*.pkl",
        "*pca*.joblib",
        "*PCA*.joblib",
        "*modelo_pca*",
        "*model_pca*",
    ]

    for pattern in patterns:
        candidates.extend(list(outcome_dir.rglob(pattern)))

    candidates = [
        p for p in candidates
        if p.is_file()
        and p.suffix.lower() in [".pkl", ".joblib", ""]
        and "variance" not in p.name.lower()
        and "csv" not in p.name.lower()
    ]

    # Deduplicate
    candidates = sorted(set(candidates), key=lambda x: len(str(x)))

    if len(candidates) == 0:
        print("\nNo PCA file found in:", outcome_dir)
        print("Files containing 'pca':")
        for p in outcome_dir.rglob("*"):
            if "pca" in p.name.lower():
                print(" ", p)
        raise FileNotFoundError(f"No PCA pickle/joblib found in {outcome_dir}")

    print("\nPCA candidates in", outcome_dir)
    for p in candidates:
        print(" ", p)

    # Prefer files inside model/ and named modelo_pca
    preferred = [
        p for p in candidates
        if "modelo_pca" in p.name.lower()
        or "model_pca" in p.name.lower()
    ]

    if preferred:
        return preferred[0]

    return candidates[0]


def load_pca(path):
    try:
        return joblib.load(path)
    except Exception as e:
        raise RuntimeError(f"Could not load PCA file {path}: {e}")


def get_feature_names(pca, features_path):
    # Best case: sklearn kept names
    if hasattr(pca, "feature_names_in_"):
        names = list(pca.feature_names_in_)
        if len(names) == getattr(pca, "n_features_in_", len(names)):
            return names

    n_features = getattr(pca, "n_features_in_", None)

    if n_features is None:
        if hasattr(pca, "components_"):
            n_features = pca.components_.shape[1]
        else:
            raise RuntimeError("Could not infer number of PCA input features.")

    features_df = pd.read_csv(features_path)

    if features_df.shape[1] == n_features:
        return list(features_df.columns)

    # Fallback: try common saved files
    print("WARNING: lista_global_vars columns do not match PCA n_features.")
    print("features_df columns:", features_df.shape[1])
    print("pca n_features:", n_features)

    return [f"feature_{i+1}" for i in range(n_features)]


def assign_domain(feature):
    f = str(feature).upper()

    if any(x in f for x in ["PAÍS", "PAIS", "ETNIA", "EDAD", "GENERO", "GÉNERO", "ORIENTSEX"]):
        return "Sociodemographic indicators"

    if "FUGAS" in f:
        return "Running away"

    if "ABUSOSUBS" in f or "SUBS" in f:
        return "Substance use"

    if "CONVIVEN" in f:
        return "Household composition"

    if "AUTOEFIC" in f:
        return "Self-efficacy"

    if "IMPULS" in f:
        return "Impulsivity"

    if "APOYO" in f:
        return "Social support"

    if "MORAL" in f:
        return "Moral domain"

    return "Other"


def compute_pca_tables(outcome, pca, feature_names):
    n_components = pca.components_.shape[0]
    n_features = pca.components_.shape[1]

    if len(feature_names) != n_features:
        raise ValueError(
            f"{outcome}: feature_names length {len(feature_names)} "
            f"does not match PCA n_features {n_features}"
        )

    explained = np.asarray(pca.explained_variance_ratio_)
    cumulative = np.cumsum(explained)

    # Variance table
    variance_df = pd.DataFrame({
        "outcome": outcome,
        "component": [f"PC{i+1}" for i in range(n_components)],
        "component_number": np.arange(1, n_components + 1),
        "explained_variance_ratio": explained,
        "explained_variance_percent": explained * 100,
        "cumulative_variance_ratio": cumulative,
        "cumulative_variance_percent": cumulative * 100,
    })

    # Loadings
    # PCA.components_ are unit vectors. Loadings can be approximated as:
    # component coefficient * sqrt(explained variance)
    if hasattr(pca, "explained_variance_"):
        loadings = pca.components_.T * np.sqrt(np.asarray(pca.explained_variance_))
    else:
        loadings = pca.components_.T

    loading_rows = []
    top_rows = []

    for pc_idx in range(n_components):
        pc_name = f"PC{pc_idx+1}"

        for feat_idx, feat in enumerate(feature_names):
            loading = float(loadings[feat_idx, pc_idx])
            abs_loading = abs(loading)

            loading_rows.append({
                "outcome": outcome,
                "component": pc_name,
                "component_number": pc_idx + 1,
                "feature": feat,
                "domain": assign_domain(feat),
                "loading": loading,
                "abs_loading": abs_loading,
                "explained_variance_ratio_component": explained[pc_idx],
            })

        pc_loadings = pd.DataFrame([
            r for r in loading_rows
            if r["component"] == pc_name
        ]).sort_values("abs_loading", ascending=False)

        for rank, (_, row) in enumerate(pc_loadings.head(TOP_N_LOADINGS_PER_PC).iterrows(), start=1):
            top_rows.append({
                "outcome": outcome,
                "component": pc_name,
                "component_number": pc_idx + 1,
                "rank": rank,
                "feature": row["feature"],
                "domain": row["domain"],
                "loading": row["loading"],
                "abs_loading": row["abs_loading"],
                "explained_variance_percent_component": explained[pc_idx] * 100,
                "cumulative_variance_percent_at_component": cumulative[pc_idx] * 100,
            })

    loadings_df = pd.DataFrame(loading_rows)
    top_loadings_df = pd.DataFrame(top_rows)

    # Approximate variance-weighted feature back-projection
    # This is not causal/model feature importance. It summarizes how strongly
    # each original variable contributes to the retained PCA representation.
    feature_summary = (
        loadings_df
        .assign(weighted_abs_loading=lambda d: d["abs_loading"] * d["explained_variance_ratio_component"])
        .groupby(["outcome", "feature", "domain"], as_index=False)
        .agg(
            total_abs_loading=("abs_loading", "sum"),
            variance_weighted_abs_loading=("weighted_abs_loading", "sum"),
            max_abs_loading=("abs_loading", "max"),
        )
    )

    total_weighted = feature_summary["variance_weighted_abs_loading"].sum()
    if total_weighted > 0:
        feature_summary["normalized_variance_weighted_contribution"] = (
            feature_summary["variance_weighted_abs_loading"] / total_weighted
        )
    else:
        feature_summary["normalized_variance_weighted_contribution"] = np.nan

    feature_summary = feature_summary.sort_values(
        ["outcome", "normalized_variance_weighted_contribution"],
        ascending=[True, False]
    )

    domain_summary = (
        feature_summary
        .groupby(["outcome", "domain"], as_index=False)
        .agg(
            variance_weighted_abs_loading=("variance_weighted_abs_loading", "sum"),
            normalized_variance_weighted_contribution=("normalized_variance_weighted_contribution", "sum"),
            n_features=("feature", "nunique"),
        )
        .sort_values(["outcome", "normalized_variance_weighted_contribution"], ascending=[True, False])
    )

    return variance_df, loadings_df, top_loadings_df, feature_summary, domain_summary


# ------------------------------------------------------------
# Run for all outcomes
# ------------------------------------------------------------

all_variance = []
all_loadings = []
all_top_loadings = []
all_feature_summary = []
all_domain_summary = []

for outcome, outcome_dir in OUTCOME_DIRS.items():
    print("\n================================================")
    print("Outcome:", outcome)
    print("Dir:", outcome_dir)
    print("Exists:", outcome_dir.exists())

    pca_path = find_pca_file(outcome_dir)
    print("Selected PCA file:", pca_path)

    pca = load_pca(pca_path)

    print("PCA type:", type(pca))
    print("PCA components shape:", pca.components_.shape)
    print("Explained variance sum:", float(np.sum(pca.explained_variance_ratio_)))

    feature_names = get_feature_names(pca, FEATURES_PATH)

    print("n feature names:", len(feature_names))
    print("first feature names:", feature_names[:10])

    variance_df, loadings_df, top_loadings_df, feature_summary_df, domain_summary_df = compute_pca_tables(
        outcome=outcome,
        pca=pca,
        feature_names=feature_names
    )

    all_variance.append(variance_df)
    all_loadings.append(loadings_df)
    all_top_loadings.append(top_loadings_df)
    all_feature_summary.append(feature_summary_df)
    all_domain_summary.append(domain_summary_df)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

variance_all = pd.concat(all_variance, ignore_index=True)
loadings_all = pd.concat(all_loadings, ignore_index=True)
top_loadings_all = pd.concat(all_top_loadings, ignore_index=True)
feature_summary_all = pd.concat(all_feature_summary, ignore_index=True)
domain_summary_all = pd.concat(all_domain_summary, ignore_index=True)

variance_all.to_csv(OUTPUT_DIR / "pca_variance_all_outcomes.csv", index=False)
loadings_all.to_csv(OUTPUT_DIR / "pca_loadings_all_outcomes.csv", index=False)
top_loadings_all.to_csv(OUTPUT_DIR / "pca_top_loadings_by_component_all_outcomes.csv", index=False)
feature_summary_all.to_csv(OUTPUT_DIR / "pca_feature_backprojection_summary_all_outcomes.csv", index=False)
domain_summary_all.to_csv(OUTPUT_DIR / "pca_domain_backprojection_summary_all_outcomes.csv", index=False)

# Pretty versions
variance_pretty = variance_all.copy()
variance_pretty["explained_variance_percent"] = variance_pretty["explained_variance_percent"].map(lambda x: f"{x:.2f}%")
variance_pretty["cumulative_variance_percent"] = variance_pretty["cumulative_variance_percent"].map(lambda x: f"{x:.2f}%")
variance_pretty.to_csv(OUTPUT_DIR / "pca_variance_all_outcomes_pretty.csv", index=False)

feature_summary_pretty = feature_summary_all.copy()
feature_summary_pretty["normalized_variance_weighted_contribution_percent"] = (
    feature_summary_pretty["normalized_variance_weighted_contribution"] * 100
).map(lambda x: f"{x:.2f}%")
feature_summary_pretty.to_csv(OUTPUT_DIR / "pca_feature_backprojection_summary_all_outcomes_pretty.csv", index=False)

domain_summary_pretty = domain_summary_all.copy()
domain_summary_pretty["normalized_variance_weighted_contribution_percent"] = (
    domain_summary_pretty["normalized_variance_weighted_contribution"] * 100
).map(lambda x: f"{x:.2f}%")
domain_summary_pretty.to_csv(OUTPUT_DIR / "pca_domain_backprojection_summary_all_outcomes_pretty.csv", index=False)


# ------------------------------------------------------------
# Display compact summaries
# ------------------------------------------------------------

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2400)

print("\n=== PCA VARIANCE SUMMARY ===")
print(
    variance_all
    .groupby("outcome")
    .agg(
        n_components=("component_number", "max"),
        retained_variance_percent=("explained_variance_percent", "sum"),
        final_cumulative_percent=("cumulative_variance_percent", "max")
    )
    .reset_index()
    .to_string(index=False, float_format=lambda x: f"{x:.2f}")
)

print("\n=== TOP 10 FEATURES BY VARIANCE-WEIGHTED BACK-PROJECTION ===")
for outcome in OUTCOME_DIRS.keys():
    print("\n", outcome)
    print(
        feature_summary_all[feature_summary_all["outcome"] == outcome]
        .head(10)
        [["feature", "domain", "normalized_variance_weighted_contribution"]]
        .to_string(index=False, float_format=lambda x: f"{x:.4f}")
    )

print("\n=== DOMAIN SUMMARY ===")
print(
    domain_summary_all
    .to_string(index=False, float_format=lambda x: f"{x:.4f}")
)

print("\nSaved outputs to:")
print(OUTPUT_DIR.resolve())

BASE_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final
FEATURES_PATH: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/data/lista_global_vars.csv True
OUTPUT_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_pca_variance_backprojection_PCA_trainonly

Outcome: victimization
Dir: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_victim/DT_victim_PCA_trainonly_TEST
Exists: True

No PCA file found in: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_victim/DT_victim_PCA_trainonly_TEST
Files containing 'pca':


FileNotFoundError: No PCA pickle/joblib found in /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_victim/DT_victim_PCA_trainonly_TEST